In [1]:
import pandas as pd
import re
import time
import logging
import xml.etree.ElementTree as ET

import requests


In [2]:
SEC_USER_AGENT = "yoyoman021@gmail.com"
SEC_HEADERS = {"User-Agent": SEC_USER_AGENT}


In [4]:
url = "https://www.sec.gov/files/company_tickers_exchange.json"
r = requests.get(url, headers=SEC_HEADERS, timeout=30)
r.raise_for_status()
data = r.json()
df = pd.DataFrame(data["data"], columns=data["fields"])
nasdaq = df[df["exchange"].str.lower() == "nasdaq"].copy()
display(nasdaq.head(2))



,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,320193,Apple Inc.,AAPL,Nasdaq


In [9]:
url = f"https://data.sec.gov/submissions/CIK0001045810.json"
r = requests.get(url, headers=SEC_HEADERS, timeout=30)
data = r.json()
recent = data.get("filings", {}).get("recent", {})
forms = recent.get("form", [])
accessions = recent.get("accessionNumber", [])
dates = recent.get("filingDate", [])
docs = recent.get("primaryDocument", [])
docs = [item.split("/")[-1] for item in docs]
filings = []
for form, acc, date, doc in zip(forms, accessions, dates, docs):
    if form == "4":
        filings.append({"cik": '0001045810', "accession": acc, "date": date, "doc": doc})

In [63]:
filings

[{'cik': '0001045810',
  'accession': '0001197647-26-000005',
  'date': '2026-07-06',
  'doc': 'wk-form4_1783371701.xml'},
 {'cik': '0001045810',
  'accession': '0001199039-26-000009',
  'date': '2026-06-29',
  'doc': 'wk-form4_1782767219.xml'},
 {'cik': '0001045810',
  'accession': '0001725292-26-000004',
  'date': '2026-06-29',
  'doc': 'wk-form4_1782767179.xml'},
 {'cik': '0001045810',
  'accession': '0001197652-26-000007',
  'date': '2026-06-29',
  'doc': 'wk-form4_1782767137.xml'},
 {'cik': '0001045810',
  'accession': '0001768670-26-000004',
  'date': '2026-06-29',
  'doc': 'wk-form4_1782767093.xml'},
 {'cik': '0001045810',
  'accession': '0001284116-26-000004',
  'date': '2026-06-29',
  'doc': 'wk-form4_1782767057.xml'},
 {'cik': '0001045810',
  'accession': '0001197650-26-000002',
  'date': '2026-06-29',
  'doc': 'wk-form4_1782767025.xml'},
 {'cik': '0001045810',
  'accession': '0001219888-26-000002',
  'date': '2026-06-29',
  'doc': 'wk-form4_1782766973.xml'},
 {'cik': '000104

In [73]:
for f in filings:
    acc_nodash = f['accession'].replace("-", "")
    base_url = f"https://www.sec.gov/Archives/edgar/data/{int(f['cik'])}/{acc_nodash}/"

    candidates = f['doc']
    if not doc.lower().endswith(".xml"):
        candidates.append(re.sub(r"\.\w+$", ".xml", doc))

    root = None
    for candidate in candidates:
        try:
            r = requests.get(base_url + candidate, headers=SEC_HEADERS, timeout=30)
            print(r.content)
            if r.status_code == 200 and b"<ownershipDocument>" in r.content:
                root = ET.fromstring(r.content)
                break
        except (requests.RequestException, ET.ParseError):
            continue
    if root is None:
        continue
    
    issuer = root.find(".//issuer")
    ticker_el = issuer.find("issuerTradingSymbol") if issuer is not None else None
    ticker = ticker_el.text.strip() if ticker_el is not None and ticker_el.text else None

    owner = root.find(".//reportingOwner")
    owner_name, is_ceo = None, False
    if owner is not None:
        name_el = owner.find(".//rptOwnerName")
        owner_name = name_el.text.strip() if name_el is not None and name_el.text else None

        rel = owner.find(".//reportingOwnerRelationship")
        if rel is not None:
            title_el = rel.find("officerTitle")
            if title_el is not None and title_el.text:
                t = title_el.text.upper()
                if "CEO" in t or "CHIEF EXECUTIVE" in t:
                    is_ceo = True

    results = []
    for txn in root.findall(".//nonDerivativeTransaction"):
        code_el = txn.find(".//transactionCode")
        code = code_el.text if code_el is not None else None
        if code != "P":  # P = open-market/private purchase
            continue

        shares_el = txn.find(".//transactionShares/value")
        shares = float(shares_el.text) if shares_el is not None and shares_el.text else None

        price_el = txn.find(".//transactionPricePerShare/value")
        price = None
        if price_el is not None and price_el.text:
            try:
                price = float(price_el.text)
            except ValueError:
                price = None

        date_el = txn.find(".//transactionDate/value")
        date = date_el.text if date_el is not None else None

        transaction_value = (shares * price) if (shares is not None and price) else None

        results.append(
            {
                "date": date,
                "shares": shares,
                "price_per_share": price,
                "transaction_value": transaction_value,
                "ticker": ticker,
                "director": owner_name,
                "market": "NASDAQ",
                "source_url": base_url + doc,
            }
        )
    print("filing done")

b'<!DOCTYPE html>\n<html lang="en">\n\n<head>\n    <meta http-equiv="X-UA-Compatible" content="IE=edge" />\n    <meta http-equiv="content-type" content="text/html; charset=utf-8">\n    <meta name="viewport" content="width=device-width initial-scale=1.0 maximum-scale=3.0" />\n    <meta name="origin" content="WCM" />\n    <script type="text/javascript">\n\t  WebFontConfig = {\n\t    google: { families: [ \'Raleway:300,400,500,600:latin\' ] }\n\t  };\n\t  (function() {\n\t    var wf = document.createElement(\'script\');\n\t    wf.src = \'//ajax.googleapis.com/ajax/libs/webfont/1/webfont.js\';\n\t    wf.type = \'text/javascript\';\n\t    wf.async = \'true\';\n\t    var s = document.getElementsByTagName(\'script\')[0];\n\t    s.parentNode.insertBefore(wf, s);\n\t  })(); </script>\n\n    <title>SEC.gov | File Unavailable</title>\n    <link rel="stylesheet" property="stylesheet" type=\'text/css\' href=\'/apology_objects/ecms/css/short-app.css\' />\n    <link rel="stylesheet" property="stylesh

KeyboardInterrupt: 

In [66]:
results

[]